In [ ]:
import pandas as pd
import numpy as np

def prepare_channel_data(df):
    # 1. Combine all individual anomaly columns into a single 'Anomaly_Bin' feature [cite: 301]
    # (Assuming your anomaly columns are named something like 'Anomaly_1', 'Anomaly_2', etc.)
    anomaly_columns = [col for col in df.columns if 'Anomaly' in col]
    
    if anomaly_columns:
        # If any specific anomaly column has a 1, Anomaly_Bin becomes 1
        df['Anomaly_Bin'] = df[anomaly_columns].max(axis=1)
        # Drop the individual anomaly columns so they aren't used as training features
        df = df.drop(columns=anomaly_columns)
    
    # 2. Drop columns with missing values [cite: 303]
    df_cleaned = df.dropna(axis=1)
    
    # 3. Separate features (X) and target (y)
    # Also dropping timestamp/identifier columns which shouldn't be used as features [cite: 236, 238]
    cols_to_drop = ['Anomaly_Bin', 'OBT', 'UTC_Timestamp']
    existing_cols_to_drop = [col for col in cols_to_drop if col in df_cleaned.columns]
    
    X = df_cleaned.drop(columns=existing_cols_to_drop)
    y = df_cleaned['Anomaly_Bin']
    
    return X, y

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score

# Load the dataset
df = pd.read_csv('flighttest_dataset.csv', low_memory=False)
channels = [15, 16, 17, 19]

# Define the models we want to compare
models = {
    "Random forest": RandomForestClassifier(random_state=42),
    "Logistic regression": LogisticRegression(max_iter=2000, random_state=42),
    "SVM (RBF Kernel)": SVC(random_state=42),
}

In [20]:
results = []

for channel in channels:
    # 1. Isolate the channel data
    df_c = df[df['Channel'] == channel].copy()
    
    # 2. Combine all individual anomalies into a single target
    anomaly_columns = [col for col in df_c.columns if 'Anomaly' in col]
    if anomaly_columns:
        df_c['Anomaly_Bin'] = df_c[anomaly_columns].max(axis=1).fillna(0)
        df_c = df_c.drop(columns=anomaly_columns)
        
    # Replace infinite values and drop columns containing missing values
    df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_cleaned = df_c.dropna(axis=1)
    
    # 3. Separate Features and Target
    cols_to_drop = ['Anomaly_Bin', 'OBT', 'UTC_Timestamp', 'Row_ID', 'Channel']
    existing_cols_to_drop = [col for col in cols_to_drop if col in df_cleaned.columns]
    
    X = df_cleaned.drop(columns=existing_cols_to_drop)
    X = X.select_dtypes(include=[np.number]) # Keep numeric only
    
    # Drop columns that exceed float32 max limits
    max_float32 = np.finfo(np.float32).max
    X_cols_to_drop = [col for col in X.columns if (X[col].abs() > max_float32).any()]
    X = X.drop(columns=X_cols_to_drop)

    y = df_c['Anomaly_Bin'].astype(int)
    
    # 4. 75/25 Stratified Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    
    # Iterate through the dictionary of models
    for model_name, model in models.items():
        
        # 5. Build Pipeline: Scaler -> PCA (95% variance) -> Classifier
        pipeline = Pipeline(steps=[
            ('scaler', StandardScaler()),
            ('pca', PCA(n_components=0.95)), 
            ('model', model)
        ])
        
        # 6. Fit and Evaluate Model
        try:
            pipeline.fit(X_train, y_train)
            y_pred = pipeline.predict(X_test)
            
            precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
            recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
            f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
            support = sum(y_test == 1)
            
            results.append({
                'Channel': channel,
                'Model': model_name,
                'Precision': f"{precision:.2f}",
                'Recall': f"{recall:.2f}",
                'F1-Score': f"{f1:.2f}",
                'Support': support
            })
        except Exception as e:
            results.append({
                'Channel': channel,
                'Model': model_name,
                'Precision': "Error",
                'Recall': "Error",
                'F1-Score': "Error",
                'Support': sum(y_test == 1)
            })

C:\Users\jayan\AppData\Local\Temp\ipykernel_9736\1867007638.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\jayan\AppData\Local\Temp\ipykernel_9736\1867007638.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\jayan\AppData\Local\Temp\ipykernel_9736\1867007638.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, exp

In [21]:
results_df = pd.DataFrame(results)

# Loop through each unique model and print its results grouped together
for model_name in models.keys():
    print(f"{model_name}:\n")
    
    # Filter the dataframe for the current model
    df_model = results_df[results_df['Model'] == model_name].copy()
    
    # Drop the 'Model' column and reset the index to 0, 1, 2, 3...
    df_model = df_model.drop(columns=['Model']).reset_index(drop=True)
    
    # Print the cleanly formatted table
    print(df_model.to_string())
    print("\n")

Random forest:

   Channel Precision Recall F1-Score  Support
0       15      0.99   0.98     0.99     2089
1       16      0.97   0.95     0.96      174
2       17      0.92   0.87     0.89      347
3       19      0.99   0.93     0.96     1725


Logistic regression:

   Channel Precision Recall F1-Score  Support
0       15      0.95   0.93     0.94     2089
1       16      0.96   0.90     0.93      174
2       17      0.78   0.88     0.82      347
3       19      0.97   0.91     0.94     1725


SVM (RBF Kernel):

   Channel Precision Recall F1-Score  Support
0       15      0.98   0.98     0.98     2089
1       16      1.00   0.90     0.95      174
2       17      0.92   0.83     0.87      347
3       19      0.99   0.93     0.96     1725




For KNN

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score

# Load the dataset
df = pd.read_csv('flighttest_dataset.csv', low_memory=False)
channels = [15, 16, 17, 19]

results = []

for channel in channels:
    # 1. Isolate the channel data
    df_c = df[df['Channel'] == channel].copy()
    
    # 2. Combine all individual anomalies into a single target
    anomaly_columns = [col for col in df_c.columns if 'Anomaly' in col]
    if anomaly_columns:
        df_c['Anomaly_Bin'] = df_c[anomaly_columns].max(axis=1).fillna(0)
        df_c = df_c.drop(columns=anomaly_columns)
        
    # Replace infinite values and drop columns containing missing values
    df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_cleaned = df_c.dropna(axis=1)
    
    # 3. Separate Features and Target
    cols_to_drop = ['Anomaly_Bin', 'OBT', 'UTC_Timestamp', 'Row_ID', 'Channel']
    existing_cols_to_drop = [col for col in cols_to_drop if col in df_cleaned.columns]
    
    X = df_cleaned.drop(columns=existing_cols_to_drop)
    X = X.select_dtypes(include=[np.number]) # Keep numeric only
    
    # Drop columns that exceed float32 max limits
    max_float32 = np.finfo(np.float32).max
    X_cols_to_drop = [col for col in X.columns if (X[col].abs() > max_float32).any()]
    X = X.drop(columns=X_cols_to_drop)

    y = df_c['Anomaly_Bin'].astype(int)
    
    # 4. 75/25 Stratified Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    
    # 5. Build Pipeline: Scaler -> PCA (95% variance) -> KNN
    pipeline = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=0.95)), 
        ('model', KNeighborsClassifier())
    ])
    
    # 6. Fit and Evaluate Model
    try:
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
        recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
        f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
        support = sum(y_test == 1)
        
        results.append({
            'Channel': channel,
            'Precision': f"{precision:.2f}",
            'Recall': f"{recall:.2f}",
            'F1-Score': f"{f1:.2f}",
            'Support': support
        })
    except Exception as e:
        print(f"FAILED on Channel {channel}. Error: {e}")
        results.append({
            'Channel': channel,
            'Precision': "Error",
            'Recall': "Error",
            'F1-Score': "Error",
            'Support': sum(y_test == 1)
        })

results_df = pd.DataFrame(results)

print("\nK-Nearest Neighbors:\n")
print(results_df.to_string())
print("\n")

C:\Users\jayan\AppData\Local\Temp\ipykernel_9736\4199628405.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\jayan\AppData\Local\Temp\ipykernel_9736\4199628405.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\jayan\AppData\Local\Temp\ipykernel_9736\4199628405.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, exp


K-Nearest Neighbors:

   Channel Precision Recall F1-Score  Support
0       15      0.99   0.99     0.99     2089
1       16      0.95   0.93     0.94      174
2       17      0.90   0.86     0.88      347
3       19      0.97   0.93     0.95     1725




Decision Tree

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score

# Load the dataset
df = pd.read_csv('flighttest_dataset.csv', low_memory=False)
channels = [15, 16, 17, 19]

results = []

for channel in channels:
    # 1. Isolate the channel data
    df_c = df[df['Channel'] == channel].copy()
    
    # 2. Combine all individual anomalies into a single target
    anomaly_columns = [col for col in df_c.columns if 'Anomaly' in col]
    if anomaly_columns:
        df_c['Anomaly_Bin'] = df_c[anomaly_columns].max(axis=1).fillna(0)
        df_c = df_c.drop(columns=anomaly_columns)
        
    # Replace infinite values and drop columns containing missing values
    df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_cleaned = df_c.dropna(axis=1)
    
    # 3. Separate Features and Target
    cols_to_drop = ['Anomaly_Bin', 'OBT', 'UTC_Timestamp', 'Row_ID', 'Channel']
    existing_cols_to_drop = [col for col in cols_to_drop if col in df_cleaned.columns]
    
    X = df_cleaned.drop(columns=existing_cols_to_drop)
    X = X.select_dtypes(include=[np.number]) # Keep numeric only
    
    # Drop columns that exceed float32 max limits
    max_float32 = np.finfo(np.float32).max
    X_cols_to_drop = [col for col in X.columns if (X[col].abs() > max_float32).any()]
    X = X.drop(columns=X_cols_to_drop)

    y = df_c['Anomaly_Bin'].astype(int)
    
    # 4. 75/25 Stratified Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    
    # 5. Build Pipeline: Scaler -> PCA (95% variance) -> Decision Tree
    pipeline = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=0.95)), 
        ('model', DecisionTreeClassifier(random_state=42))
    ])
    
    # 6. Fit and Evaluate Model
    try:
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        
        precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
        recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
        f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
        support = sum(y_test == 1)

        dt_model = pipeline.named_steps['model']
        
        # The root node is always at index 0 in the tree's arrays
        root_feature_index = dt_model.tree_.feature[0]
        root_gini = dt_model.tree_.impurity[0]
        
        print(f"--- Channel {channel} ---")
        print(f"Root Node Feature: PCA Component {root_feature_index}")
        print(f"Root Node Gini Index: {root_gini:.4f}\n")

        y_pred = pipeline.predict(X_test)
        
        results.append({
            'Channel': channel,
            'Precision': f"{precision:.2f}",
            'Recall': f"{recall:.2f}",
            'F1-Score': f"{f1:.2f}",
            'Support': support
        })
    except Exception as e:
        print(f"FAILED on Channel {channel}. Error: {e}")
        results.append({
            'Channel': channel,
            'Precision': "Error",
            'Recall': "Error",
            'F1-Score': "Error",
            'Support': sum(y_test == 1)
        })

results_df = pd.DataFrame(results)

print("\nDecision Tree:\n")
print(results_df.to_string())
print("\n")

--- Channel 15 ---
Root Node Feature: PCA Component 0
Root Node Gini Index: 0.4044

--- Channel 16 ---
Root Node Feature: PCA Component 0
Root Node Gini Index: 0.4049



C:\Users\jayan\AppData\Local\Temp\ipykernel_34116\2084433757.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\jayan\AppData\Local\Temp\ipykernel_34116\2084433757.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_c.replace([np.inf, -np.inf], np.nan, inplace=True)


--- Channel 17 ---
Root Node Feature: PCA Component 1
Root Node Gini Index: 0.4048



C:\Users\jayan\AppData\Local\Temp\ipykernel_34116\2084433757.py:27: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_c.replace([np.inf, -np.inf], np.nan, inplace=True)


--- Channel 19 ---
Root Node Feature: PCA Component 0
Root Node Gini Index: 0.4050


Decision Tree:

   Channel Precision Recall F1-Score  Support
0       15      0.97   0.98     0.98     2089
1       16      0.97   0.97     0.97      174
2       17      0.88   0.84     0.86      347
3       19      0.92   0.94     0.93     1725




WITHOUT PCA

In [2]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv('flighttest_dataset.csv', low_memory=False)
channels = [15, 16, 17, 19]

for channel in channels:
    # 1. Isolate the channel data
    df_c = df[df['Channel'] == channel].copy()
    
    # 2. Combine all individual anomalies into a single target
    anomaly_columns = [col for col in df_c.columns if 'Anomaly' in col]
    if anomaly_columns:
        df_c['Anomaly_Bin'] = df_c[anomaly_columns].max(axis=1).fillna(0)
        df_c = df_c.drop(columns=anomaly_columns)
        
    # Replace infinite values and drop columns containing missing values
    df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_cleaned = df_c.dropna(axis=1)
    
    # 3. Separate Features and Target
    cols_to_drop = ['Anomaly_Bin', 'OBT', 'UTC_Timestamp', 'Row_ID', 'Channel']
    existing_cols_to_drop = [col for col in cols_to_drop if col in df_cleaned.columns]
    
    X = df_cleaned.drop(columns=existing_cols_to_drop)
    X = X.select_dtypes(include=[np.number]) # Keep numeric only
    
    # Drop columns that exceed float32 max limits
    max_float32 = np.finfo(np.float32).max
    X_cols_to_drop = [col for col in X.columns if (X[col].abs() > max_float32).any()]
    X = X.drop(columns=X_cols_to_drop)

    y = df_c['Anomaly_Bin'].astype(int)
    
    # 4. 75/25 Stratified Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    
    # 5. Fit Decision Tree DIRECTLY (No PCA)
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    
    # 6. Extract Tree Information for the Root Node
    tree = model.tree_
    
    # Get the feature index and name used at the root (node 0)
    root_feature_idx = tree.feature[0]
    root_feature_name = X_train.columns[root_feature_idx]
    
    # Get Gini impurity and sample counts for the root
    root_gini = tree.impurity[0]
    n_node_samples = tree.n_node_samples
    samples_total = n_node_samples[0]
    
    # Get child nodes to calculate the Information Gain (Impurity Reduction)
    left_child = tree.children_left[0]
    right_child = tree.children_right[0]
    
    gini_left = tree.impurity[left_child]
    gini_right = tree.impurity[right_child]
    
    samples_left = n_node_samples[left_child]
    samples_right = n_node_samples[right_child]
    
    # Calculate the weighted impurity of the children
    weighted_child_gini = (samples_left / samples_total) * gini_left + (samples_right / samples_total) * gini_right
    
    # Information Gain = Impurity before split - Weighted Impurity after split
    impurity_reduction = root_gini - weighted_child_gini
    
    # Print the exact formatted string
    print(f"Channel {channel}:")
    print(f"The root node uses the feature “{root_feature_name}”. Its Gini index before splitting is {root_gini:.12f}. The split reduces impurity by {impurity_reduction:.12f}.")
    print("-" * 40)

Channel 15:
The root node uses the feature “payload.EMOD.DP_ResetCounter”. Its Gini index before splitting is 0.404413509384. The split reduces impurity by 0.315897362809.
----------------------------------------
Channel 16:
The root node uses the feature “payload.GMOD.DP_FlashRolloverCounter”. Its Gini index before splitting is 0.404948216882. The split reduces impurity by 0.304497343190.
----------------------------------------
Channel 17:
The root node uses the feature “param readonly uint16 totalFixFailures”. Its Gini index before splitting is 0.404778324859. The split reduces impurity by 0.250230356007.
----------------------------------------


C:\Users\jayan\AppData\Local\Temp\ipykernel_34116\772892240.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\jayan\AppData\Local\Temp\ipykernel_34116\772892240.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_c.replace([np.inf, -np.inf], np.nan, inplace=True)
C:\Users\jayan\AppData\Local\Temp\ipykernel_34116\772892240.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, exp

Channel 19:
The root node uses the feature “platform.EPS.expectedSwitchStatesBitmap”. Its Gini index before splitting is 0.404986968013. The split reduces impurity by 0.303977158418.
----------------------------------------


Random forest gave relatively better results compared to other algorithms.